# Train Your Own Reasoning Model with NeMo AutoModel

This tutorial shows how to train a language model that can **selectively
enable or disable chain-of-thought reasoning** based on the system prompt.
The trained model will:

* Produce step-by-step reasoning inside `<think>...</think>` tags when the
  system prompt says **"detailed thinking on"**.
* Answer directly (no chain-of-thought) when the system prompt says
  **"detailed thinking off"**.

This controllable-reasoning approach is inspired by the
[Llama Nemotron](https://huggingface.co/nvidia/Llama-3_1-Nemotron-Ultra-253B-v1)
family of models and follows the methodology outlined in the
NVIDIA blog post
["Train Your Own Reasoning Model in 48 Hours"](https://developer.nvidia.com/blog/train-your-own-reasoning-model/).

We use **Llama 3.1 8B Instruct** as the base model and support two
training modes:

| Mode | Hardware | Time (estimate) | Config |
|------|----------|-----------------|--------|
| **LoRA** | 1x H100 80 GB | ~48 hours | `reasoning_sft_lora_config.yaml` |
| **Full Fine-Tuning** | 8x H100 80 GB | ~12 hours | `reasoning_sft_full_config.yaml` |

---
## Prerequisites

* **NeMo AutoModel** installed (see the
  [AutoModel README](https://github.com/NVIDIA-NeMo/Automodel) for
  installation instructions).
* **Hardware**: 1x H100 80 GB GPU for LoRA, or 8x H100 80 GB for full
  fine-tuning.
* **Hugging Face token** with access to
  [meta-llama/Llama-3.1-8B-Instruct](https://huggingface.co/meta-llama/Llama-3.1-8B-Instruct).
* **Training data** in OpenAI chat format (JSONL), with system prompts
  that toggle reasoning on or off.

---
## Step 0 -- Environment Setup

If you are running inside the NeMo AutoModel container, the environment is already
configured. Otherwise, launch the container with:

```bash
docker run --gpus all -it --rm \
  -v $(pwd):/workspace \
  -w /workspace \
  --shm-size=16g \
  nvcr.io/nvidia/nemo-automodel:26.02.00 \
  bash
```

Verify that the `automodel` CLI is available:

In [ ]:
!automodel --help | head -5

---
## Step 1 -- Prepare Data

The training data must be in **OpenAI chat format** (JSONL), where each
line is a JSON object with a `messages` array. The **system prompt**
controls whether the assistant should reason step-by-step or answer
directly.

Below are examples of both modes.

In [ ]:
import json

# Example: reasoning enabled
reasoning_on = {
    "messages": [
        {"role": "system", "content": "detailed thinking on"},
        {"role": "user", "content": "Solve: If a train travels at 60 mph for 2.5 hours, how far does it go?"},
        {"role": "assistant", "content": "<think>\nTo find the distance, I need to multiply speed by time.\nSpeed = 60 mph\nTime = 2.5 hours\nDistance = 60 * 2.5 = 150 miles\n</think>\n\nThe train travels 150 miles."}
    ]
}

# Example: reasoning disabled (direct answer)
reasoning_off = {
    "messages": [
        {"role": "system", "content": "detailed thinking off"},
        {"role": "user", "content": "What is the capital of France?"},
        {"role": "assistant", "content": "The capital of France is Paris."}
    ]
}

print("Reasoning ON example:")
print(json.dumps(reasoning_on, indent=2))
print("\nReasoning OFF example:")
print(json.dumps(reasoning_off, indent=2))

### Data Preparation Tips

* **Volume**: You need at least **2000 training steps** at a global batch
  size of 256 to observe reliable reasoning behavior. Plan your dataset
  size accordingly.
* **Mix**: Include both *reasoning-on* and *reasoning-off* samples so the
  model learns when to reason and when to answer directly.
* **Curriculum ordering**: Sorting samples from easy to hard (e.g., by
  question difficulty or response length) can improve convergence.
* **Batch size**: A global batch size of **256** is recommended for stable
  training. Gradient accumulation handles the difference between global
  and local batch sizes.
* **Context length**: Use at least **8 192 tokens** to accommodate
  chain-of-thought responses. Set `packed_sequence_size: 8192` in the
  config to enable sequence packing for better throughput.
* **Output file**: Save your dataset as a JSONL file (e.g.,
  `training.jsonl`) and update the `path_or_dataset_id` field in the
  YAML config.

---
## Step 2 -- Choose Training Mode

Select between **LoRA** (parameter-efficient, single GPU) and **full
fine-tuning** (all weights, multi-GPU).

In [ ]:
# Configuration
DO_LORA = True  # Set False for full fine-tuning
CONFIG_PATH = (
    "tutorials/reasoning-sft/reasoning_sft_lora_config.yaml"
    if DO_LORA
    else "tutorials/reasoning-sft/reasoning_sft_full_config.yaml"
)
print(f"Training mode: {'LoRA' if DO_LORA else 'Full Fine-Tuning'}")
print(f"Config: {CONFIG_PATH}")

---
## Step 3 -- Review the Configuration

Let's inspect the YAML config to understand each section before
launching training.

In [ ]:
from pathlib import Path

print(Path(CONFIG_PATH).read_text())

### Key Configuration Parameters

| Parameter | LoRA | Full FT | Why |
|---|---|---|---|
| `peft.dim` | 64 | N/A | Higher LoRA rank (64+) captures complex reasoning patterns |
| `peft.alpha` | 128 | N/A | 2x dim for stable LoRA training |
| `optimizer.lr` | 1e-4 | 5e-6 | LoRA tolerates a higher learning rate |
| `distributed.tp_size` | 1 | 2 | Full FT needs tensor parallelism for memory |
| `step_scheduler.max_steps` | 2000 | 2000 | Minimum steps to observe reasoning behavior |
| `step_scheduler.global_batch_size` | 256 | 256 | Large batch size for stable training |
| `lr_scheduler.lr_warmup_steps` | 100 | 100 | Warm up avoids early instability |
| `packed_sequence.packed_sequence_size` | 0 | 0 | Set to 8192 for production throughput |

---
## Step 4 -- Launch Training

Run the `automodel` CLI with the selected config. For full fine-tuning,
we pass `--nproc-per-node 8` to use all 8 GPUs.

In [ ]:
if DO_LORA:
    !automodel tutorials/reasoning-sft/reasoning_sft_lora_config.yaml
else:
    !automodel tutorials/reasoning-sft/reasoning_sft_full_config.yaml --nproc-per-node 8

---
## Step 5 -- Monitor Training

During training, watch for these signals:

* **Loss curve**: The loss should decrease from ~2+ down to below 1.0
  over the course of training.
* **Curriculum effect**: If you ordered your data from easy to hard, you
  may see a loss spike when the model transitions to harder examples,
  followed by a drop near the end of the epoch.
* **Validation loss**: Should track the training loss downward. If
  validation loss plateaus while training loss keeps decreasing, the
  model may be overfitting.

To enable W&B logging, uncomment the `wandb` section in the YAML config.
You can also monitor training with TensorBoard if configured in your
environment.

---
## Step 6 -- Test the Reasoning Model

Load the latest checkpoint and test both reasoning modes. The
consolidated checkpoint is saved in Hugging Face SafeTensors format and
can be loaded directly with `transformers`.

In [ ]:
import glob
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

CKPT_BASE = "checkpoints/reasoning_sft/" if DO_LORA else "checkpoints/reasoning_sft_full/"
ckpt_dirs = sorted(glob.glob(f"{CKPT_BASE}epoch_*_step_*"))
latest = ckpt_dirs[-1] + "/model/consolidated" if ckpt_dirs else None
print(f"Loading checkpoint: {latest}")

tokenizer = AutoTokenizer.from_pretrained(latest or "meta-llama/Llama-3.1-8B-Instruct")
model = AutoModelForCausalLM.from_pretrained(
    latest or "meta-llama/Llama-3.1-8B-Instruct",
    torch_dtype=torch.bfloat16,
    device_map="auto",
)
model.eval()


def generate_response(messages, max_new_tokens=512):
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        output = model.generate(**inputs, max_new_tokens=max_new_tokens, temperature=0.7, do_sample=True)
    return tokenizer.decode(output[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)


# Test with reasoning ON
print("=" * 60)
print("REASONING ON")
print("=" * 60)
response = generate_response([
    {"role": "system", "content": "detailed thinking on"},
    {"role": "user", "content": "A store has 3 apples. They receive 2 shipments of 5 apples each. How many apples total?"},
])
print(response)

# Test with reasoning OFF
print("\n" + "=" * 60)
print("REASONING OFF")
print("=" * 60)
response = generate_response([
    {"role": "system", "content": "detailed thinking off"},
    {"role": "user", "content": "What is 2 + 2?"},
])
print(response)

---
## Step 7 -- Evaluate

Run a quantitative evaluation on GSM8K (grade school math) to measure
reasoning accuracy. This uses the
[lm-evaluation-harness](https://github.com/EleutherAI/lm-evaluation-harness)
with a vLLM backend.

In [ ]:
import glob

ckpt_dirs = sorted(glob.glob(f"{CKPT_BASE}epoch_*_step_*"))
eval_ckpt = ckpt_dirs[-1] + "/model/consolidated" if ckpt_dirs else "meta-llama/Llama-3.1-8B-Instruct"
print(f"Evaluating: {eval_ckpt}")

!lm_eval --model vllm \
    --model_args pretrained={eval_ckpt},dtype=auto \
    --tasks gsm8k \
    --batch_size auto \
    --apply_chat_template \
    --output_path results/gsm8k

---
## Step 8 (Optional) -- Export for Deployment

The consolidated checkpoint is already saved in Hugging Face format
(SafeTensors + config) and can be deployed directly with
[vLLM](https://docs.vllm.ai/) or
[NVIDIA NIM](https://developer.nvidia.com/nim).

In [ ]:
# The consolidated checkpoint is already HF-compatible.
# Deploy with vLLM:
# vllm serve checkpoints/reasoning_sft/latest/model/consolidated --dtype auto

# Or with NVIDIA NIM:
# See https://developer.nvidia.com/nim for deployment options
print("Checkpoint is ready for deployment!")
print(f"Path: {latest}")

---
## Next Steps

* **More benchmarks** -- Evaluate on MATH, ARC, MMLU, and other
  reasoning benchmarks. See the [Evaluation tutorial](../evaluation/)
  for details.
* **RLHF / DPO** -- To further improve reasoning quality beyond SFT,
  apply reinforcement learning from human feedback.
  See [NeMo-RL](https://github.com/NVIDIA/NeMo-RL).
* **Longer training** -- Increase `max_steps` beyond 2000 for
  stronger reasoning. Many successful reasoning models train for
  5000--10000+ steps.
* **LoRA rank** -- Experiment with higher LoRA ranks (128, 256) for
  richer reasoning representations, at the cost of more memory.
* **Curriculum ordering** -- Sort training data from easy to hard
  problems to improve convergence and final accuracy.

---
## Resources

* [NeMo AutoModel](https://github.com/NVIDIA-NeMo/Automodel) -- the
  training framework used in this tutorial.
* [NeMo Curator](https://github.com/NVIDIA/NeMo-Curator) -- tools for
  large-scale data curation and preprocessing.
* [NeMo-RL](https://github.com/NVIDIA/NeMo-RL) -- RLHF, DPO, and PPO
  alignment pipelines.
* [Hugging Face Transformers](https://huggingface.co/docs/transformers) --
  for checkpoint loading and inference.
* [Llama 3.1 8B Instruct](https://huggingface.co/meta-llama/Llama-3.1-8B-Instruct) --
  the base model used in this tutorial.
* [lm-evaluation-harness](https://github.com/EleutherAI/lm-evaluation-harness) --
  for benchmark evaluation.
* [vLLM](https://docs.vllm.ai/) -- high-throughput inference engine.